# Mobile Price Prediction
### Analysing Features That Determine Mobile Phone Price Ranges

## 1. Project Overview
This notebook explores a dataset of mobile phone specifications to understand which hardware and connectivity features influence price classification. The dataset contains 2,000 phones categorised into four price ranges (0–3, low to very high).

## 2. Learning Objectives
- Explore distributions of mobile hardware features
- Identify which specs (RAM, battery, camera) correlate with price range
- Perform statistical tests across price categories
- Visualise feature importances with multi-panel charts
- Understand classification-oriented EDA

## 3. Business / Research Problem
**Question:** Which mobile phone specifications are the strongest indicators of price range? Can we identify clear feature thresholds that separate low-cost from premium devices?

## 4. Why This Analysis Matters
Mobile manufacturers and retailers need to understand how hardware specifications map to price tiers. This insight guides product design, competitive positioning, and marketing strategy.

## 5. Dataset Overview
Key features include:
- `battery_power` — battery capacity (mAh)
- `blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, `wifi` — binary features
- `clock_speed` — processor speed
- `fc`, `pc` — front and primary camera megapixels
- `int_memory` — internal memory (GB)
- `ram` — RAM (MB)
- `px_height`, `px_width` — screen resolution
- `sc_h`, `sc_w` — screen dimensions
- `talk_time` — battery talk time
- `mobile_wt` — weight
- `n_cores` — processor cores
- `price_range` — target (0=low, 1=medium, 2=high, 3=very high)

## 6. Dataset Source and License Notes
- **Source:** Mobile price classification dataset
- **Format:** CSV with 21 features + target
- **License:** Public / educational use

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
TRAIN_FILE = DATA_DIR / 'train.csv'
TEST_FILE = DATA_DIR / 'test.csv'
PRICE_LABELS = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very High'}

## 10. Dataset Loading

In [4]:
df = pd.read_csv(TRAIN_FILE)
print(f'Train shape: {df.shape}')
df.head()

Train shape: (2000, 21)


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


## 11. Data Validation Checks

In [5]:
print('Columns:', df.columns.tolist())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\nMissing values:', df.isnull().sum().sum())
print('\nTarget distribution:')
print(df['price_range'].value_counts().sort_index())
print('\nBasic stats:')
df.describe().T

Columns: ['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g', 'touch_screen', 'wifi', 'price_range']

Duplicates: 0

Missing values: 0

Target distribution:
price_range
0    500
1    500
2    500
3    500
Name: count, dtype: int64

Basic stats:


,count,mean,std,min,25%,50%,75%,max
battery_power,2000.0,1238.51850,439.418206,501.0,851.75,1226.0,1615.25,1998.0
blue,2000.0,0.49500,0.500100,0.0,0.00,0.0,1.00,1.0
clock_speed,2000.0,1.52225,0.816004,0.5,0.70,1.5,2.20,3.0
dual_sim,2000.0,0.50950,0.500035,0.0,0.00,1.0,1.00,1.0
fc,2000.0,4.30950,4.341444,0.0,1.00,3.0,7.00,19.0
four_g,2000.0,0.52150,0.499662,0.0,0.00,1.0,1.00,1.0
int_memory,2000.0,32.04650,18.145715,2.0,16.00,32.0,48.00,64.0
m_dep,2000.0,0.50175,0.288416,0.1,0.20,0.5,0.80,1.0
mobile_wt,2000.0,140.24900,35.399655,80.0,109.00,141.0,170.00,200.0
n_cores,2000.0,4.52050,2.287837,1.0,3.00,4.0,7.00,8.0


## 12. Data Cleaning
The dataset is already clean with no missing values. We add readable price labels.

In [6]:
df['price_label'] = df['price_range'].map(PRICE_LABELS)
print('Price labels added.')
print(df['price_label'].value_counts())

Price labels added.
price_label
Medium       500
High         500
Very High    500
Low          500
Name: count, dtype: int64


## 13. Exploratory Data Analysis

In [7]:
print('RAM statistics by price range:')
print(df.groupby('price_range')['ram'].describe())
print(f'\nOverall feature ranges:')
for col in ['battery_power','ram','int_memory','px_height','px_width']:
    print(f'  {col}: {df[col].min()} — {df[col].max()}')

RAM statistics by price range:
             count      mean         std     min      25%     50%      75%  \
price_range                                                                  
0            500.0   785.314  362.755397   256.0   488.75   719.5  1037.00   
1            500.0  1679.490  465.850159   387.0  1354.00  1686.5  2033.75   
2            500.0  2582.816  496.211735  1185.0  2292.75  2577.0  2927.00   
3            500.0  3449.232  393.019923  2259.0  3207.50  3509.5  3764.75   

                max  
price_range          
0            1974.0  
1            2811.0  
2            3916.0  
3            3998.0  

Overall feature ranges:
  battery_power: 501 — 1998
  ram: 256 — 3998
  int_memory: 2 — 64
  px_height: 0 — 1960
  px_width: 500 — 1998


## 14. Univariate Analysis

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['ram'], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('RAM Distribution')

axes[0,1].hist(df['battery_power'], bins=40, color='coral', edgecolor='white')
axes[0,1].set_title('Battery Power Distribution')

axes[0,2].hist(df['int_memory'], bins=30, color='seagreen', edgecolor='white')
axes[0,2].set_title('Internal Memory Distribution')

df['price_range'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color=['#2a9d8f','#457b9d','#e9c46a','#e76f51'])
axes[1,0].set_title('Price Range Distribution')
axes[1,0].tick_params(axis='x', rotation=0)

axes[1,1].hist(df['px_height'], bins=40, color='purple', edgecolor='white')
axes[1,1].set_title('Pixel Height Distribution')

binary_cols = ['blue','dual_sim','four_g','three_g','touch_screen','wifi']
binary_means = df[binary_cols].mean()
binary_means.plot(kind='bar', ax=axes[1,2], color='teal')
axes[1,2].set_title('Binary Feature Prevalence')
axes[1,2].set_ylabel('Proportion')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='price_range', y='ram', ax=axes[0], palette='viridis')
axes[0].set_title('RAM by Price Range')

sns.boxplot(data=df, x='price_range', y='battery_power', ax=axes[1], palette='viridis')
axes[1].set_title('Battery Power by Price Range')

sns.boxplot(data=df, x='price_range', y='px_height', ax=axes[2], palette='viridis')
axes[2].set_title('Pixel Height by Price Range')

plt.tight_layout(); plt.show()

In [10]:
# RAM vs Battery colored by price
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='ram', y='battery_power', hue='price_range',
                palette='viridis', alpha=0.5, ax=ax)
ax.set_title('RAM vs Battery Power by Price Range')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Correlation analysis — which features separate price tiers?

In [11]:
corr_with_price = df.select_dtypes(include=[np.number]).corr()['price_range'].drop('price_range').sort_values(ascending=False)
print('Feature correlations with price_range:')
print(corr_with_price.to_string())

fig, ax = plt.subplots(figsize=(8, 8))
corr_with_price.plot(kind='barh', ax=ax, color=corr_with_price.apply(lambda x: 'steelblue' if x > 0 else 'coral'))
ax.set_title('Feature Correlation with Price Range')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()

Feature correlations with price_range:
ram              0.917046
battery_power    0.200723
px_width         0.165818
px_height        0.148858
int_memory       0.044435
sc_w             0.038711
pc               0.033599
three_g          0.023611
sc_h             0.022986
fc               0.021998
talk_time        0.021859
blue             0.020573
wifi             0.018785
dual_sim         0.017444
four_g           0.014772
n_cores          0.004399
m_dep            0.000853
clock_speed     -0.006606
mobile_wt       -0.030302
touch_screen    -0.030411


## 17. Statistical Checks / Hypothesis Testing
**Test:** ANOVA — does RAM differ significantly across price ranges?

In [12]:
groups = [g['ram'].values for _, g in df.groupby('price_range')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA for RAM across price ranges:')
print(f'F={f_stat:.2f}, p={p_val:.2e}')
print(f'Significant: {p_val < 0.05}')

print('\nMean RAM by price range:')
print(df.groupby('price_range')['ram'].mean().to_string())

ANOVA for RAM across price ranges:
F=3520.11, p=0.00e+00
Significant: True

Mean RAM by price range:
price_range
0     785.314
1    1679.490
2    2582.816
3    3449.232


## 18. Visual Analysis
### Feature distributions across price tiers

In [13]:
features = ['ram','battery_power','px_height','int_memory']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, features):
    for pr in sorted(df['price_range'].unique()):
        subset = df[df['price_range'] == pr][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=PRICE_LABELS[pr])
    ax.set_title(f'{feat} by Price Range')
    ax.legend()
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **RAM is the strongest predictor** of price range (r ≈ 0.92) — a near-perfect separator.
2. **Battery power** shows moderate correlation with price range.
3. **Binary features** (WiFi, Bluetooth, 4G) are evenly distributed and weakly predictive.
4. **Screen resolution** (px_height, px_width) correlates positively with price.
5. **Internal memory** has surprisingly low correlation with price range.

## 20. Limitations
- Synthetic dataset — relationships may be artificially clean.
- Price range is categorical (0–3), not actual currency values.
- No brand, OS, or market data included.
- Feature interactions may be overly simplified.

## 21. Recommendations / Next Steps
1. Build classification models (logistic regression, random forest, SVM) using these features.
2. Feature engineering: total screen pixels = px_height × px_width.
3. Evaluate whether RAM alone achieves >90% classification accuracy.
4. Compare train and test set distributions for data drift.
5. Apply PCA to reduce the 20 features and visualise clusters.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Treating price_range as continuous | It is an ordinal categorical variable | Use classification, not regression |
| Ignoring RAM's dominance | It overwhelms other features | Analyse with and without RAM |
| Not scaling features | Features have vastly different ranges | Apply StandardScaler |
| Using all binary features | They add noise with low signal | Feature selection first |
| Leaking test data | Using test.csv for EDA | Only explore train.csv |

## 23. Mini Challenge / Exercises
1. **Feature engineering:** Create `screen_area = sc_h × sc_w` and `total_pixels = px_height × px_width`. Do these improve separability?
2. **Pair plot:** Make a pairplot of the top 4 correlated features colored by price_range.
3. **Chi-square test:** Test whether `four_g` is independent of price_range.
4. **Decision boundary:** With just RAM, can you define thresholds that separate all 4 classes?
5. **Dimensionality reduction:** Apply PCA to reduce to 2D and plot clusters.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  RAM is the dominant feature separating mobile price ranges.
TAKEAWAY 2  Battery power and screen resolution are secondary predictors.
TAKEAWAY 3  Binary connectivity features have minimal predictive power.
TAKEAWAY 4  The dataset is balanced across all four price categories.
TAKEAWAY 5  Simple feature thresholds on RAM alone can approximate price classification.
```